# Stream Processing — Four Ways to Consume a Data Stream

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ankush-Chander/DS614-big-data-engineering/blob/main/docs/notebooks/stream_processing.ipynb)

> *"Batch processing is answering questions about the past; stream processing is answering questions about the present."* — Jay Kreps

In this notebook we'll build and observe **four different streaming consumers**, progressing from the simplest (polling a file) to cloud-managed queues (SQS).

Each approach solves the same problem — reading a continuous flow of **taxi-trip events** — but makes different trade-offs in latency, reliability, and complexity.

| # | Technique | Key Concept | Real-world Analogy |
|---|-----------|------------|-------------------|
| 1 | **Polling** | Pull-based; consumer asks "anything new?" | Refreshing your email inbox |
| 2 | **Unix Pipes & Sockets** | OS-level push; kernel notifies consumer | `tail -f` on a log file |
| 3 | **Redis Pub/Sub & RabbitMQ** | Broker-mediated messaging | Post office / bulletin board |
| 4 | **Amazon SQS** | Managed cloud queue | Cloud-native microservices |

##  Prerequisites

### Python packages
```bash
pip install redis pika boto3
```

### External services (via Docker)
```bash
# Redis
docker run -d --name redis-demo -p 6379:6379 redis:7

# RabbitMQ (with management UI at http://localhost:15672, guest/guest)
docker run -d --name rabbitmq-demo -p 5672:5672 -p 15672:15672 rabbitmq:3-management

# LocalStack (AWS SQS emulator)
docker run -d --name localstack -p 4566:4566 localstack/localstack
```

### Helper script
All demos use `resources/event_generator.py` — a shared producer that generates fake taxi-trip events and sends them to various sinks.

---
## Polling

### What is Polling?
The consumer **periodically checks** a data source for new records.

**Analogy:** Checking your physical mailbox every 5 minutes.

### Architecture

```
┌──────────┐      writes       ┌──────────────────┐
│ Producer  │ ───────────────▶  │ File / Database   │
└──────────┘                   └────────┬─────────┘
                                        │ reads every N sec
                                        ▼
                               ┌──────────────────┐
                               │    Consumer       │
                               │  (Polling Loop)   │
                               └──────────────────┘
```

**Key Parameters:**
- **Polling interval** — how often do we check?
- **Cursor / offset** — how do we know what's "new"?

In [1]:
import subprocess, os

# Clean slate
open("/tmp/stream_events.jsonl", "w").close()

# Start producer in background: writes 30 events, 1 per second
producer = subprocess.Popen(
    ["python", "resources/event_generator.py", "file",
     "--path", "/tmp/stream_events.jsonl",
     "--interval", "1", "--count", "30"],
)
print(f"✅ Producer started (PID={producer.pid}), writing events to /tmp/stream_events.jsonl")

✅ Producer started (PID=8859), writing events to /tmp/stream_events.jsonl


In [2]:
import json, time

def polling_consumer(filepath, poll_interval=2, max_polls=10):
    """
    Reads new lines from a file by tracking the byte offset.
    This is the simplest possible streaming consumer.
    """
    offset = 0
    events_seen = 0
    
    for poll_num in range(1, max_polls + 1):
        with open(filepath, "r") as f:
            f.seek(offset)
            new_lines = f.readlines()
            offset = f.tell()   # remember where we left off
        
        if new_lines:
            for line in new_lines:
                event = json.loads(line)
                events_seen += 1
                print(f"  [Poll #{poll_num}] Event {events_seen}: "
                      f"{event['pickup_zone']} | ${event['fare_estimate']}")
        else:
            print(f"  [Poll #{poll_num}] No new events.")
        
        time.sleep(poll_interval)    # ⏳ wait before next poll
    
    print(f"\n✅ Polling complete. Total events consumed: {events_seen}")

polling_consumer("/tmp/stream_events.jsonl")

  [Poll #1] Event 1: Staten Island | $18.4
  [Poll #1] Event 2: Brooklyn | $46.83
  [Poll #1] Event 3: Manhattan | $34.05
  [Poll #1] Event 4: Queens | $77.82
  [Poll #1] Event 5: Staten Island | $49.0
  [Poll #1] Event 6: Staten Island | $16.81
  [Poll #1] Event 7: Manhattan | $19.7
  [Poll #1] Event 8: Manhattan | $26.37
  [Poll #1] Event 9: Brooklyn | $18.72
  [Poll #1] Event 10: Bronx | $32.54
  [Poll #1] Event 11: Bronx | $73.39
  [Poll #1] Event 12: Queens | $32.92
  [Poll #1] Event 13: Manhattan | $28.14
  [Poll #1] Event 14: Brooklyn | $61.4
  [Poll #1] Event 15: Manhattan | $81.79
  [Poll #2] Event 16: Manhattan | $82.09
  [Poll #2] Event 17: Queens | $83.0
  [Poll #3] Event 18: Brooklyn | $39.77
  [Poll #3] Event 19: Queens | $6.14
  [Poll #4] Event 20: Queens | $28.94
  [Poll #4] Event 21: Queens | $53.08
  [Poll #5] Event 22: Brooklyn | $73.01
  [Poll #5] Event 23: Brooklyn | $78.35
  [Poll #6] Event 24: Staten Island | $49.65
  [Poll #6] Event 25: Brooklyn | $15.85
  [Poll

In [3]:
# Clean up the producer process
producer.terminate()
producer.wait()
print("🧹 Producer stopped.")

🧹 Producer stopped.


###  Observations

| Aspect | Observation |
|--------|------------|
| **Latency** | Up to `poll_interval` seconds of delay |
| **CPU usage** | Wasted cycles when there's nothing new |
| **Ordering** | Guaranteed (sequential file reads) |
| **Fault tolerance** | Offset can be persisted for recovery |

### Limitations
- **Trade-off:** Short interval = more CPU; long interval = higher latency.
- **No push notification** — consumer is blind between polls.
- **Scalability:** Multiple consumers need coordination (locks, partitions).

### When to Use
Polling is appropriate when:
- The data source doesn't support push (e.g., REST APIs, legacy databases)
- Latency requirements are relaxed (seconds to minutes)
- Simplicity is more important than efficiency

---
## Unix Pipes & Sockets

### Watching with the OS
Instead of asking "is there anything new?", we let the **operating system tell us** when data arrives. This is **event-driven / push-based** at the OS level.

We'll explore two sub-approaches:

| Mechanism | How It Works |
|-----------|-------------|
| `tail -f` + pipe | OS watches file inode for appends |
| TCP Socket | Kernel delivers bytes as they arrive |

### Architecture (Socket)
```
┌──────────┐    TCP stream     ┌──────────────────┐
│ Producer  │ ════════════════▶ │  Socket Consumer  │
└──────────┘   push-based      └──────────────────┘
               (kernel-managed)
```

### Demo A: `tail -f` via subprocess

The `tail -f` command follows a file and outputs new lines as they are appended.  
We pipe its stdout into Python — the OS handles the "waiting" for us.

On Linux, tail -F uses inotify (kernel event system).

### What is inotify?

A Linux API that lets programs subscribe to filesystem events.
Instead of polling:  
Kernel notifies when something changes  

#### How tail -F works
1. Watch file + parent directory  
2. If file is modified → read new lines  
3. If file is deleted/moved:  
a. Stop reading old inode   
b. Wait for new file with same name    
4.  When new file appears:    
a. Open new inode   
b. Continue tailing   

In [6]:
import subprocess, json, time

# Ensure fresh file
open("/tmp/stream_events.jsonl", "w").close()

# Start producer: writes 15 events at 1/sec
print(" ".join(["python", "resources/event_generator.py", "file", "--path", "/tmp/stream_events.jsonl", "--interval", "1", "--count", "15"]))
producer = subprocess.Popen(
    ["python", "resources/event_generator.py", "file",
     "--path", "/tmp/stream_events.jsonl",
     "--interval", "1", "--count", "15"],
)

# Consumer: tail -f piped into our Python process
# tail -f /tmp/stream_events.jsonl
tail_proc = subprocess.Popen(
    ["tail", "-f", "/tmp/stream_events.jsonl"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

events_seen = 0
try:
    for line in tail_proc.stdout:
        line = line.strip()
        if not line:
            continue
        event = json.loads(line)
        events_seen += 1
        print(f"  [tail -f] Event {events_seen}: "
              f"{event['pickup_zone']} | ${event['fare_estimate']}")
        if events_seen >= 15:
            break
finally:
    tail_proc.terminate()
    producer.wait()

print(f"\n✅ tail -f complete. Events consumed: {events_seen}")

python resources/event_generator.py file --path /tmp/stream_events.jsonl --interval 1 --count 15
  [tail -f] Event 1: Manhattan | $22.77
  [tail -f] Event 2: Staten Island | $36.23
  [tail -f] Event 3: Staten Island | $23.73
  [tail -f] Event 4: Staten Island | $72.67
  [tail -f] Event 5: Staten Island | $31.18
  [tail -f] Event 6: Staten Island | $65.14
  [tail -f] Event 7: Bronx | $58.53
  [tail -f] Event 8: Queens | $33.9
  [tail -f] Event 9: Bronx | $51.43
  [tail -f] Event 10: Bronx | $53.85
  [tail -f] Event 11: Staten Island | $58.91
  [tail -f] Event 12: Manhattan | $28.75
  [tail -f] Event 13: Brooklyn | $83.01
  [tail -f] Event 14: Staten Island | $55.74
  [tail -f] Event 15: Queens | $5.78
🚀 Starting event generator: sink=file, interval=1.0s, count=15
  [file] Wrote event 1/15
  [file] Wrote event 2/15
  [file] Wrote event 3/15
  [file] Wrote event 4/15
  [file] Wrote event 5/15
  [file] Wrote event 6/15
  [file] Wrote event 7/15
  [file] Wrote event 8/15
  [file] Wrote even

### Demo B: TCP Socket Consumer

A socket server listens for incoming connections. The producer connects and pushes events directly — no file intermediary, no polling interval.

In [ ]:
import socket, threading, json, time, subprocess

def start_socket_server(host="localhost", port=9999, max_events=15):
    """
    Listens on a TCP socket and prints events as they arrive.
    The OS kernel handles buffering and notification.
    """
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((host, port))
    server.listen(1)
    server.settimeout(30)  # don't hang forever
    print(f"🔌 Socket server listening on {host}:{port}")
    
    conn, addr = server.accept()
    print(f"   Connected by {addr}")
    
    buffer = ""
    events_seen = 0
    
    try:
        while events_seen < max_events:
            data = conn.recv(4096).decode()
            if not data:
                break
            buffer += data
            # Process complete lines (newline-delimited JSON)
            while "\n" in buffer:
                line, buffer = buffer.split("\n", 1)
                if line.strip():
                    event = json.loads(line)
                    events_seen += 1
                    print(f"  [Socket] Event {events_seen}: "
                          f"{event['pickup_zone']} | ${event['fare_estimate']}")
    finally:
        conn.close()
        server.close()
    
    print(f"\n✅ Socket consumer done. Events: {events_seen}")


# Run server in a thread so the notebook doesn't block
server_thread = threading.Thread(target=start_socket_server)
server_thread.start()
time.sleep(1)  # give server a moment to bind

# Start producer targeting the socket
producer = subprocess.Popen(
    ["python", "resources/event_generator.py", "socket",
     "--host", "localhost", "--port", "9999",
     "--interval", "1", "--count", "15"],
)

server_thread.join(timeout=30)
producer.wait()

### Observations

| Aspect | `tail -f` / Pipe | TCP Socket |
|--------|-----------------|------------|
| **Latency** | Near-zero (kernel inotify) | Near-zero |
| **Push-based?** | ✅ Yes | ✅ Yes |
| **Ordering** | ✅ Single file = ordered | ✅ Single connection = ordered |
| **Multi-consumer** | ❌ Tricky | Possible (one conn per consumer) |
| **Cross-machine** | ❌ Local only | ✅ Works over network |

### Limitations
- **No persistence:** If the consumer is down, messages are lost (sockets) or pile up (file).
- **No replay:** Can't re-read old messages easily.
- **Tight coupling:** Producer and consumer must agree on protocol.

### Key Takeaway
OS-level mechanisms give us **low latency** but **zero reliability guarantees**.  
This is where **message brokers** come in…

---
## Message Brokers: Redis & RabbitMQ

### Why a Broker?
A **message broker** sits between producers and consumers, providing:
- **Decoupling** — producer doesn't need to know who consumes
- **Buffering** — messages are held if consumers are slow
- **Fan-out** — one message can reach many consumers

We'll demo two popular brokers with very different philosophies:

| Broker | Model | Durability | Best For |
|--------|-------|-----------|----------|
| **Redis Pub/Sub** | Fire-and-forget broadcast | ❌ Messages lost if no subscriber | Real-time notifications, caches |
| **RabbitMQ** | Durable queue with acks | ✅ Messages persist until consumed | Task queues, reliable delivery |

### Architecture
```
                          ┌───────────┐
                     ┌──▶ │Consumer A │
┌──────────┐   pub  │    └───────────┘
│ Producer  │───────▶│ BROKER
└──────────┘        │    ┌───────────┐
                     └──▶ │Consumer B │
                          └───────────┘
```

### 3a. Redis Pub/Sub

Redis Pub/Sub is a **broadcast** model: every subscriber gets every message.  
But if **no one is listening, the message is gone forever**.

In [7]:
import redis
import json, threading, time, subprocess

def redis_consumer(channel="taxi_events", max_events=15):
    """Subscribe to a Redis channel and print events as they arrive."""
    r = redis.Redis()
    pubsub = r.pubsub()
    pubsub.subscribe(channel)
    print(f"📡 Subscribed to Redis channel: '{channel}'")
    
    events_seen = 0
    for message in pubsub.listen():
        if message["type"] == "message":
            event = json.loads(message["data"])
            events_seen += 1
            print(f"  [Redis] Event {events_seen}: "
                  f"{event['pickup_zone']} | ${event['fare_estimate']}")
            if events_seen >= max_events:
                break
    
    pubsub.unsubscribe()
    print(f"\n✅ Redis consumer done. Events: {events_seen}")


# Start consumer in a thread (must subscribe BEFORE producer publishes)
consumer_thread = threading.Thread(target=redis_consumer)
consumer_thread.start()
time.sleep(1)  # ensure subscription is active

# Start producer
producer = subprocess.Popen(
    ["python", "resources/event_generator.py", "redis",
     "--channel", "taxi_events",
     "--interval", "1", "--count", "15"],
)

consumer_thread.join(timeout=30)
producer.wait()

📡 Subscribed to Redis channel: 'taxi_events'
  [Redis] Event 1: Queens | $72.74
  [Redis] Event 2: Bronx | $55.72
  [Redis] Event 3: Manhattan | $74.38
  [Redis] Event 4: Manhattan | $72.08
  [Redis] Event 5: Manhattan | $57.07
  [Redis] Event 6: Staten Island | $38.26
  [Redis] Event 7: Bronx | $30.44
  [Redis] Event 8: Staten Island | $8.39
  [Redis] Event 9: Bronx | $59.88
  [Redis] Event 10: Staten Island | $75.14
  [Redis] Event 11: Queens | $80.07
  [Redis] Event 12: Staten Island | $57.42
  [Redis] Event 13: Staten Island | $82.53
  [Redis] Event 14: Brooklyn | $43.4
  [Redis] Event 15: Staten Island | $22.23

✅ Redis consumer done. Events: 15
🚀 Starting event generator: sink=redis, interval=1.0s, count=15
  [redis] Published event 1/15
  [redis] Published event 2/15
  [redis] Published event 3/15
  [redis] Published event 4/15
  [redis] Published event 5/15
  [redis] Published event 6/15
  [redis] Published event 7/15
  [redis] Published event 8/15
  [redis] Published event 9/1

0

#### Fire-and-Forget Demo

What happens if we publish messages **before** anyone subscribes?

In [ ]:
import redis, json

r = redis.Redis()

# ── Step 1: Publish 5 events with NO subscriber listening ─────────
print("Publishing 5 events with NO subscriber active...")
for i in range(5):
    event = {"event_id": i, "note": "nobody is listening"}
    r.publish("taxi_events", json.dumps(event))
    print(f"  Published event {i}")

# ── Step 2: NOW subscribe ────────────────────────────────────────
print("\nNow subscribing...")
pubsub = r.pubsub()
pubsub.subscribe("taxi_events")

# ── Step 3: Try to read — only get the subscription confirmation ─
msg = pubsub.get_message(timeout=2)
print(f"First message type: '{msg['type']}'")  # 'subscribe', NOT 'message'

msg = pubsub.get_message(timeout=2)
print(f"Next message: {msg}")  # None — the 5 events are gone forever!

pubsub.unsubscribe()
print("\n⚠️  All 5 events were LOST because no subscriber was active!")
print("   This is the 'fire-and-forget' nature of Redis Pub/Sub.")

### 3b. RabbitMQ

RabbitMQ is a **durable queue**: messages persist until a consumer explicitly **acknowledges** them.  

To prove durability, we'll start the producer **first**, let messages pile up, and **then** start the consumer 5 seconds later.

In [ ]:
import pika
import json, subprocess, time

def rabbitmq_consumer(queue="taxi_events", max_events=15):
    """Consume from a RabbitMQ queue with manual acknowledgment."""
    connection = pika.BlockingConnection(
        pika.ConnectionParameters("localhost")
    )
    channel = connection.channel()
    channel.queue_declare(queue=queue)
    print(f"🐇 Listening on RabbitMQ queue: '{queue}'")
    
    events_seen = 0
    
    def on_message(ch, method, properties, body):
        nonlocal events_seen
        event = json.loads(body)
        events_seen += 1
        print(f"  [RabbitMQ] Event {events_seen}: "
              f"{event['pickup_zone']} | ${event['fare_estimate']}")
        ch.basic_ack(delivery_tag=method.delivery_tag)  # 👈 acknowledge
        if events_seen >= max_events:
            ch.stop_consuming()
    
    channel.basic_consume(queue=queue, on_message_callback=on_message)
    
    try:
        channel.start_consuming()
    except Exception:
        pass
    finally:
        connection.close()
    
    print(f"\n✅ RabbitMQ consumer done. Events: {events_seen}")


# ── Start producer FIRST — messages will QUEUE (unlike Redis!) ────
producer = subprocess.Popen(
    ["python", "resources/event_generator.py", "rabbitmq",
     "--queue", "taxi_events",
     "--interval", "0.5", "--count", "15"],
)

time.sleep(5)  # Let messages accumulate in the queue

# ── Start consumer — it picks up the BUFFERED messages! ──────────
print("⏳ Consumer starting 5 seconds AFTER producer...")
print("   Messages should already be waiting in the queue!\n")
rabbitmq_consumer(max_events=15)
producer.wait()

### Redis vs RabbitMQ — Head to Head

| Feature | Redis Pub/Sub | RabbitMQ |
|---------|-------------|----------|
| **Message persistence** | ❌ Fire-and-forget | ✅ Durable queues |
| **Acknowledgments** | ❌ No | ✅ Manual/auto ack |
| **Replay** | ❌ Not possible | ❌ Not built-in (consumed = gone) |
| **Throughput** | 🚀 Very high | 🏎️ High |
| **Fan-out** | ✅ All subscribers get every message | ✅ Via exchanges (fanout, topic, direct) |
| **Consumer offline** | ⚠️ Misses messages | ✅ Messages wait in queue |
| **Use case** | Real-time dashboards, caching | Task queues, order processing |

### Key Takeaway
- Use **Redis Pub/Sub** when speed matters and occasional message loss is acceptable.
- Use **RabbitMQ** when **every message must be processed** (at-least-once delivery).
- For **replay / rewind** capability, you need a **log-based broker** like Apache Kafka (out of scope today).

---
## Amazon SQS (Simple Queue Service)

### Managed Cloud Queues
SQS removes the burden of **operating your own broker infrastructure**.

Key properties:
- **Fully managed** — no servers to provision
- **At-least-once delivery** — messages are delivered at least once
- **Visibility timeout** — message is "invisible" while being processed
- **Dead-letter queues** — failed messages are routed for investigation
- **Auto-scaling** — from 1 to millions of messages/sec

### Architecture
```
┌──────────┐    SendMessage     ┌───────────┐    ReceiveMessage    ┌──────────────┐
│ Producer  │ ─────────────────▶│  AWS SQS   │◀────────────────── │   Consumer    │
└──────────┘                    │  (Queue)   │  DeleteMessage      └──────────────┘
                                └───────────┘
```

> 🧪 **We use [LocalStack](https://localstack.cloud/) to emulate SQS locally.**  
> No AWS account or charges required!

### Step 1: Create the SQS Queue

In [ ]:
import boto3, json

# Connect to LocalStack's SQS emulator
sqs = boto3.client(
    "sqs",
    endpoint_url="http://localhost:4566",
    region_name="us-east-1",
    aws_access_key_id="test",
    aws_secret_access_key="test",
)

# Create the queue
response = sqs.create_queue(
    QueueName="taxi-events",
    Attributes={
        "VisibilityTimeout": "30",          # seconds a message is hidden after receive
        "MessageRetentionPeriod": "86400",  # keep messages for 1 day
    },
)
queue_url = response["QueueUrl"]
print(f"📬 Queue created: {queue_url}")

### Step 2: Producer — Send Messages

In [ ]:
import time, random
from datetime import datetime

ZONES = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

for i in range(15):
    event = {
        "event_id": random.randint(100000, 999999),
        "timestamp": datetime.now().isoformat(),
        "pickup_zone": random.choice(ZONES),
        "passengers": random.randint(1, 6),
        "fare_estimate": round(random.uniform(5.0, 85.0), 2),
    }
    sqs.send_message(
        QueueUrl=queue_url,
        MessageBody=json.dumps(event),
        MessageAttributes={
            "EventType": {
                "DataType": "String",
                "StringValue": "trip_start",
            }
        },
    )
    print(f"  📤 Sent event {i+1}: {event['pickup_zone']} | ${event['fare_estimate']}")
    time.sleep(0.5)

print(f"\n✅ All 15 messages sent to SQS.")

### Step 3: Consumer — Long Polling

SQS supports **long polling** (`WaitTimeSeconds > 0`), which blocks the request until messages arrive (or times out). This reduces empty responses and API costs compared to short polling.

In [ ]:
events_processed = 0
max_events = 15
max_retries = 5
empty_count = 0

print(f"📥 Starting SQS consumer (long polling)...\n")

while events_processed < max_events and empty_count < max_retries:
    response = sqs.receive_message(
        QueueUrl=queue_url,
        MaxNumberOfMessages=5,        # batch up to 5
        WaitTimeSeconds=5,            # 👈 long polling (blocks up to 5s)
        MessageAttributeNames=["All"],
    )
    
    messages = response.get("Messages", [])
    
    if not messages:
        empty_count += 1
        print(f"  ⏳ No messages available (attempt {empty_count}/{max_retries}), retrying...")
        continue
    
    empty_count = 0  # reset on successful receive
    
    for msg in messages:
        event = json.loads(msg["Body"])
        events_processed += 1
        print(f"  [SQS] Event {events_processed}: "
              f"{event['pickup_zone']} | ${event['fare_estimate']}")
        
        # ✅ Delete after successful processing (acknowledge)
        sqs.delete_message(
            QueueUrl=queue_url,
            ReceiptHandle=msg["ReceiptHandle"],
        )

print(f"\n✅ SQS consumer done. Events processed: {events_processed}")

### Visibility Timeout Demo

When a consumer receives a message, SQS makes it **invisible** for `VisibilityTimeout` seconds.  
If the consumer doesn't delete it in time (e.g., it crashes), the message **re-appears** for another consumer.

In [ ]:
# ── Demonstrate the "visibility timeout" concept ──────────────────

# 1. Send one test message
sqs.send_message(
    QueueUrl=queue_url,
    MessageBody=json.dumps({"test": "visibility_timeout_demo"}),
)
print("1️⃣  Sent a test message.")

# 2. Receive it (starts the visibility timeout)
resp = sqs.receive_message(
    QueueUrl=queue_url, MaxNumberOfMessages=1, WaitTimeSeconds=5
)
msg = resp["Messages"][0]
print(f"2️⃣  Received: {msg['Body']}")
print(f"   Receipt handle: {msg['ReceiptHandle'][:40]}...")

# 3. DON'T delete it — simulate a consumer crash
print("3️⃣  Simulating consumer crash (NOT deleting the message)...")
print(f"   Message is now INVISIBLE for 30 seconds (VisibilityTimeout).")

# 4. Try to receive again immediately — nothing available!
resp2 = sqs.receive_message(
    QueueUrl=queue_url, MaxNumberOfMessages=1, WaitTimeSeconds=2
)
available = len(resp2.get("Messages", []))
print(f"4️⃣  Immediate re-read: {available} messages available (invisible!)")

print("\n💡 After the VisibilityTimeout expires, the message becomes visible again")
print("   and can be picked up by another consumer.")
print("   This is how SQS ensures AT-LEAST-ONCE delivery!")

# Clean up
sqs.delete_message(QueueUrl=queue_url, ReceiptHandle=msg["ReceiptHandle"])
print("\n🧹 Test message cleaned up.")

### SQS Key Concepts

| Concept | Description |
|---------|------------|
| **Standard Queue** | At-least-once delivery, best-effort ordering |
| **FIFO Queue** | Exactly-once processing, strict ordering (lower throughput) |
| **Long Polling** | `WaitTimeSeconds > 0` — reduces empty responses and costs |
| **Visibility Timeout** | Message is hidden while being processed; reappears if not deleted |
| **Dead Letter Queue** | Automatically routes messages that fail N times |

### Limitations
- **At-least-once delivery** means consumers **must be idempotent**
- **No real-time push** — still fundamentally polling (long-polling mitigates this)
- **Max message size** is 256 KB (use S3 for larger payloads)
- **Vendor lock-in** — tightly coupled to AWS ecosystem

---
##  Grand Comparison

| Feature | Polling | Unix Pipes/Sockets | Redis Pub/Sub | RabbitMQ | Amazon SQS |
|---------|---------|-------------------|---------------|----------|------------|
| **Model** | Pull | Push | Push (broadcast) | Push (queue) | Pull (long-poll) |
| **Latency** | ⏱️ High (interval) | ⚡ Very low | ⚡ Very low | ⚡ Low | ⏱️ Low–Med |
| **Persistence** | ✅ (file/DB) | ❌ | ❌ | ✅ | ✅ |
| **Reliability** | 🟡 DIY | ❌ Fragile | ❌ Fire-and-forget | ✅ Acks | ✅ At-least-once |
| **Multi-consumer** | 🟡 Manual | 🟡 Manual | ✅ Built-in | ✅ Built-in | ✅ Built-in |
| **Cross-machine** | ✅ (shared storage) | ✅ (sockets) | ✅ | ✅ | ✅ |
| **Complexity** | 🟢 Trivial | 🟡 Medium | 🟡 Medium | 🟠 Higher | 🟢 Low (managed) |
| **Best for** | Simple / legacy | Low-latency local | Real-time broadcast | Reliable task queues | Cloud-native apps |

## Key Takeaways

1. **Polling** is the simplest but least efficient — good for prototyping or where push isn't available.
2. **OS-level streaming** (pipes/sockets) gives ultra-low latency but no durability or fault tolerance.
3. **Message brokers** (Redis, RabbitMQ) add decoupling and fan-out, but you must operate the infrastructure.
4. **Managed services** (SQS) offload operations to the cloud provider — ideal for production workloads.
5. The right choice depends on your **latency**, **reliability**, and **operational** requirements.

> As data systems evolve, you'll often combine multiple approaches:  
> *Polling for legacy sources → brokers for internal services → SQS/Kafka for production.*

## Cleanup

Run the cell below to tear down all demo infrastructure.

In [ ]:
import subprocess, os

# Remove temp files
if os.path.exists("/tmp/stream_events.jsonl"):
    os.remove("/tmp/stream_events.jsonl")
    print("🗑️  Removed /tmp/stream_events.jsonl")

# Stop Docker containers (if running)
for name in ["redis-demo", "rabbitmq-demo", "localstack"]:
    result = subprocess.run(["docker", "stop", name], capture_output=True, text=True)
    subprocess.run(["docker", "rm", name], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"🗑️  Stopped & removed container: {name}")
    else:
        print(f"⏭️  Container '{name}' was not running.")

print("\n✅ Cleanup complete!")